In [ ]:
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import os

class StockDataPipeline:
    '''
    Automated pipeline for stock data
    '''
    def __init__(self, tickers, data_folder='stock_data'):
        self.tickers = tickers
        self.data_folder = data_folder

        # Create data folder if it doesn't exist
        if not os.path.exists(data_folder):
            os.makedirs(data_folder)

    def download_data(self, period='1y'):
        '''
        Download data for all tickers
        '''
        print(f'Downloading data for {len(self.tickers)} stocks...')

        for ticker in self.tickers:
            try:
                print(f'  Fetching {ticker}...')
                data = yf.download(ticker, period=period, progress=False)

                if not data.empty:
                    # Save to CSV
                    filename = f'{self.data_folder}/{ticker}.csv'
                    data.to_csv(filename)
                    print(f'  ✓ Saved {ticker} ({len(data)} rows)')
                else:
                    print(f'  ✗ No data for {ticker}')

            except Exception as e:
                print(f'  ✗ Error with {ticker}: {str(e)}')

        print('Download complete!')

    def load_data(self, ticker):
        ''' 
        Load data for a specific ticker
        '''
        filename = f'{self.data_folder}/{ticker}.csv'

        if os.path.exists(filename):
            data = pd.read_csv(filename, index_col=0, parse_dates=True)
            return data
        else:
            print(f'No data found for {ticker}')
            return None
        
    def update_data(self):
        '''
        Update existing data with new records
        '''
        print('Updating data...')

        for ticker in self.tickers:
            try:
                # Load existing data
                existing = self.load_data(ticker)

                if existing is not None:
                    # Get last date in existing data
                    last_date = pd.to_datetime(existing.index[-1])

                    # Download data from last date to now
                    new_data = yf.download(
                        ticker,
                        start=last_date + timedelta(days=1),
                        progress=False
                    )

                    if not new_data.empty:
                        # Combine old and new data
                        updated = pd.concat([existing, new_data])
                        updated = updated[~updated.index.duplicated(keep='last')]

                        # Save updated data
                        filename = f'{self.data_folder}/{ticker}.csv'
                        updated.to_csv(filename)
                        print(f'  ✓ Updated {ticker} (+{len(new_data)} rows)')
                    else:
                        print(f'  - {ticker} already up to date')
            except Exception as e:
                print(f'  ✗ Error updating {ticker}: {str(e)}')

        print('Update complete!')

    def get_portfolio_data(self):
        ''' 
        Load all stock data into a single DataFrame
        '''
        all_data = {}

        for ticker in self.tickers:
            data = self.load_data(ticker)
            if data is not None:
                all_data[ticker] = data['Close']

        # Combine into single DataFrame
        portfolio_df = pd.DataFrame(all_data)
        return portfolio_df
    
# Example usage
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']
pipeline = StockDataPipeline(tickers)

# Initial download
pipeline.download_data(period='2y')

# Load specific stock
aapl_data = pipeline.load_data('AAPL')
print(aapl_data.tail())

# Get all stocks in one DataFrame
portfolio = pipeline.get_portfolio_data()
print(portfolio.head())

# Update data (run this periodically)
pipeline.update_data()